In [6]:
# West Michigan Uniform
import numpy as np
import pandas as pd
import glob
import os
base_path = r"C:\Users\ageglio\OneDrive - TRC\Documents\My EQuIS Work\Mdn01\West Michigan Uniform"
out_name = "Groundwater_Elevations_Wildman.csv"

In [7]:
# sys_loc_codes = ['MW-122B', 'MW-106R', 'MW-1R', 'MW-105R', 'MW-104R', 'MW-103R', 'MW-102R', 'MW-115SR', 'MW-30R', 'MW-115DR', 'MW-116S', 'MW-116I', 'MW-5R', 'MW-3R', 'MW-101DR', 'MW-101IR', 'MW-101SR', 'MW-111R', 'MW-126DR', 'MW-41R', 'MW-118IR', 'MW-129D', 'MW-16R', 'MW-15R', 'MW-112R', 'MW-132', 'MW-37R', 'MW-40R', 'MW-19R', 'MW-7R', 'MW-10R', 'MW-114R', 'MW-22', 'MW-113R', 'MW-120S', 'MW-6R', 'MW-26R', 'MW-20', 'MW-18', 'MW-124D', 'MW-11R', 'MW-12R', 'MW-36R', 'MW-23R', 'MW-24', 'MW-25', 'MW-27', 'MW-28', 'MW-31', 'MW-32', 'MW-33', 'MW-13R', 'MW-14R']

In [8]:
dt_well_segment = pd.read_excel(os.path.join(base_path, "dt_well_segment.xlsx"))
dt_well_segment2 = dt_well_segment[['sys_loc_code', 'start_depth', 'end_depth']]

dt_water_level = pd.read_excel(os.path.join(base_path, "dt_water_level.xlsx"))
dt_water_level2 = dt_water_level[['sys_loc_code', 'measurement_date', 'water_level_depth', 'measured_depth_of_well', 'depth_unit']]

dt_coordinate = pd.read_excel(os.path.join(base_path, "dt_coordinate.xlsx"))
dt_coordinate2 = dt_coordinate[['sys_loc_code', 'x_coord', 'y_coord', 'elev']]
dt_coordinate2 = dt_coordinate2.rename(columns={"elev":"surf_elev"})

dt_measure_dataum = pd.read_excel(os.path.join(base_path, "dt_measure_datum.xlsx"))
dt_measure_dataum2 = dt_measure_dataum[['sys_loc_code', 'datum_value', 'datum_unit']]
dt_measure_dataum2 = dt_measure_dataum2.rename(columns={"datum_value":"reference_elev"})

df_combined = pd.merge(dt_water_level2, dt_well_segment2, on="sys_loc_code", how="outer")
df_combined = pd.merge(df_combined, dt_coordinate2, on="sys_loc_code", how="outer")
df_combined = pd.merge(df_combined, dt_measure_dataum2, on="sys_loc_code", how="outer")

# filter if needed
# df_combined = df_combined.loc[df_combined['sys_loc_code'].isin(sys_loc_codes)]

# Calculated fields
df_combined['stickup_calc'] = df_combined['reference_elev'] - df_combined['surf_elev']
df_combined['tos_ft_btoc'] = df_combined['start_depth'] + df_combined['stickup_calc']
df_combined['bos_ft_btoc'] = df_combined['end_depth'] + df_combined['stickup_calc']
df_combined['water_level'] = df_combined['reference_elev'] - df_combined['water_level_depth']

df_combined = df_combined[['sys_loc_code', 'x_coord', 'y_coord', 'surf_elev', 'reference_elev', 'stickup_calc', 
                           'start_depth', 'end_depth', 'tos_ft_btoc', 'bos_ft_btoc', 
                            'water_level_depth', 'water_level', 'measurement_date', 'depth_unit']]

In [9]:
df_combined

,sys_loc_code,x_coord,y_coord,surf_elev,reference_elev,stickup_calc,start_depth,end_depth,tos_ft_btoc,bos_ft_btoc,water_level_depth,water_level,measurement_date,depth_unit
0,MW-25-01,1.265095e+07,472428.431,604.143,603.732,-0.411,9.0,14.0,8.589,13.589,9.14,594.592,2025-06-13 09:25:00,ft
1,MW-25-01D,1.265094e+07,472428.578,604.322,603.869,-0.453,25.0,30.0,24.547,29.547,9.27,594.599,2025-06-13 08:40:00,ft
2,MW-25-02,1.265105e+07,472427.534,604.554,604.101,-0.453,9.0,14.0,8.547,13.547,9.30,594.801,2025-06-13 11:31:00,ft
3,MW-25-02D,1.265106e+07,472427.472,604.564,604.131,-0.433,20.0,25.0,19.567,24.567,9.35,594.781,2025-06-13 11:46:00,ft
4,MW-25-04,1.265120e+07,472350.569,603.558,602.945,-0.613,9.0,14.0,8.387,13.387,7.82,595.125,2025-06-13 10:21:00,ft
5,MW-25-06,1.265116e+07,472214.113,601.091,600.603,-0.488,5.0,10.0,4.512,9.512,5.38,595.223,2025-06-12 10:27:00,ft
6,MW-25-07,1.265106e+07,472177.710,600.713,600.485,-0.228,5.0,10.0,4.772,9.772,5.38,595.105,2025-06-12 12:40:00,ft
7,MW-25-07D,1.265106e+07,472177.880,600.728,600.244,-0.484,20.0,25.0,19.516,24.516,5.14,595.104,2025-06-12 13:13:00,ft
8,MW-25-08,1.265093e+07,472178.978,600.956,600.672,-0.284,5.0,10.0,4.716,9.716,5.79,594.882,2025-06-13 11:01:00,ft
9,MW-25-09D,1.265091e+07,472332.433,602.140,601.743,-0.397,25.0,30.0,24.603,29.603,7.25,594.493,2025-06-13 07:58:00,ft


In [10]:
df_combined.to_csv(os.path.join(base_path, out_name), index=False)